<a href="https://colab.research.google.com/github/zzuupp/Object_Detection-Segmentation/blob/main/Panoptic_Segmentation/panoptic_segmentation_on_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
print(sys.version)

3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [2]:
!pip -q install -U transformers accelerate safetensors
!pip -q install -U opencv-python-headless pillow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 157.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 152.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 171.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 141.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Pillow 버전 낮추기. 11.x..
!pip -q uninstall -y pillow
!pip -q install "pillow>=10.4,<12.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 66.5 MB/s eta 0:00:00


In [4]:
import PIL
print("Pillow:", PIL.__version__)

Pillow: 11.2.1


In [5]:
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation
print('transformers + mask2former import OK')

transformers + mask2former import OK


#### 허깅스페이스에 올라온 모델 사용.

In [6]:
import torch

# 허깅페이스에 올라온 사전학습모델 사용.
MODEL_ID = "facebook/mask2former-swin-base-coco-panoptic"  # COCO panoptic pretrained :contentReference[oaicite:2]{index=2}

# GPU가 있다면 사용할 것.
device = 'cuda'if torch.cuda.is_available() else 'cpu'

# 이미지 전처리 담당자 불러오기. (Hugging Face는 모델에 맞는 processor를 자동으로 제공)
processor = AutoImageProcessor.from_pretrained(MODEL_ID) # 모델이 이미지를 이해할 수 있는 장치 생성 (내부적으로 resize, normalize, tesor 변환)

# 자체 모델 로드 -> Huggingpace 서버에서 모델 구조, 학습된 가중치(weights) 전부 다운로드, PyTorch 모델로 로드, GPU/CPU로 이동
model = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_ID).to(device)

# 추론(inference) 모드 ⭕️ -> dropout 꺼짐 / batchnorm 고정 / 결과 안정화 + 속도 향상
model.eval()

# 숫자 라벨 → 사람이 읽는 이름 // ex) {0: 'person', 1: 'bicycle', 2: 'car', ...}
id2label = model.config.id2label
print('Loaded', MODEL_ID, 'on', device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.11/dist-packages/transformers/image_processing_base.py:417: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/432M [00:00<?, ?B/s]

Loaded facebook/mask2former-swin-base-coco-panoptic on cuda


#### panoptic_overlay_bgr() 함수 생성.
> OpenCV로 읽은 영상 프레임(BGR, NumPy 배열)을 받아서</br>
Panoptic segmentation 결과를 색으로 덧입힌 프레임을 반환하는 함수

In [7]:
import numpy as np
import cv2
from PIL import Image

# “OpenCV로 읽은 영상 프레임(BGR, NumPy 배열)을 받아서 Panoptic segmentation 결과를 색으로 덧입힌 프레임을 반환하는 함수”
def panoptic_overlay_bgr(frame_bgr : np.ndarray, alpha : float = 0.55):
    """
    -----
    OpenCV(cv2.VideoCapture, cv2.imread)가 반환하는 영상 프레임은 무조건 NumPy 배열.
    또한 OpenCV는 BGR 로 채널 순서가 되어 있으므로 명시함. *OpenCV BGR frame(H, W, 3)
    alpha:  0.5 근처가 원본도 보이고, segmentation도 보이는 가장 가독성 좋은 값.
    -----
    """

    # BGR -> RGB PIL // 대부분의 PIL / HuggingFace / 대부분의 DL 모델 규칙 (RGB)
    image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # NumPy 배열을 “RGB 이미지로 해석될 수 있는 PIL Image 객체로 변환 //array로부터 이미지 객체 생성.
    pil = Image.fromarray(image_rgb)


    # 정규화처리 및 텐서 변환 이후, GPU 등의 장치로 이동.
    # return_tensors : 전처리 결과를 어떠한 자료형으로 반환할 것인가 (우리는 pytorch 형식)
    inputs = processor(images = pil, return_tensors = 'pt').to(device)

    # 기울기 업데이트 불필요.
    with torch.no_grad():
        outputs = model(**inputs) # ** 에 대해서 생각해보기.

    # pil.size = (width, height)와 같은 형태 // segmentation 결과를 원본 크기로 되돌리기 위해 (h,w) 필요
    # [::-1] 순서를 뒤집는다는 뜻 ([start = 끝 : stop = 처음 : step = -1]) 처음부터 끝까지 거꾸로 읽어라.
    target_size = [pil.size[::-1]]

    # post_process_panoptic_segmentation은 batch 단위 결과를 리스트로 반환한다.
    # 현재는 단일 이미지(batch size=1)이므로 첫 번째(유일한) 결과(dict)를 사용한다.
    panoptic = processor.post_process_panoptic_segmentation(
        outputs,
        target_sizes=target_size
    )[0]

    '''
    ● [post-process 내부 동작 요약]
    - outputs에서 mask logits 추출
    - softmax / sigmoid 등으로 확률화
    - confidence threshold 적용
    - mask 간 충돌 해결 (overlap resolution)
    - instance별 id 부여
    - stuff 영역 병합
    - 최종적으로:
        - panoptic["segmentation"] : (H, W) segment id map 생성
        - panoptic["segments_info"]: 각 segment id에 대한 메타정보 생성
    👉 segmentation map은 "모델이 직접 출력"하는 것이 아니라, 이 post-process 단계에서 처음 생성된다.


    ● [반환값 구조]
    List[Dict[str, Any]]
    ex)
    [
        {
            "segmentation": np.ndarray (H, W),   # segment id map
            "segments_info": [                  # 각 segment id의 메타정보
                {"id": int, "label_id": int, "score": float, "isthing": bool},
                ...
            ]
        }
    ]


    ● [segmentation map 해석]
    panoptic["segmentation"] 은 (H, W) 형태의 2D 정수 map이며,
    각 픽셀 값은 "클래스 번호"가 아니라 "segment id"이다.
    → 같은 클래스라도 인스턴스가 다르면 서로 다른 id를 가짐 (panoptic의 핵심)

    (segmentation map 개념 예시)
    0 0 0 0 0
    0 1 1 1 0
    0 1 1 1 0
    0 2 2 0 0
    0 2 2 0 0

    이때 id=0/1/2가 무엇을 의미하는지는 panoptic["segments_info"]를 조회해서 해석한다.
    (예: id=0 → road(stuff), id=1 → person#1(thing), id=2 → car#1(thing))


    ● [시각화 필요성]
    segment id map은 사람이 직접 해석하기 어렵기 때문에,
    각 segment id에 고유한 색상을 부여하여 "컬러 마스크"로 변환한 뒤,
    원본 프레임과 alpha blending하여 overlay로 시각화한다.
    '''



    # panoptic segmentation 결과(픽셀별 segment id)를 GPU에 있는 PyTorch 텐서 → CPU NumPy 배열 → 정수형 지도(map)로 변환
    seg_map = panoptic["segmentation"].cpu().numpy().astype(np.int32)


    # 세그먼트(id)별 “설명서” 목록
    segments_info = panoptic['segments_info']

    ''' 여기서 segments_info 리스트 예시
    ex)
            [
        {"id": 1, "label_id": 0, "isthing": True,  "area": 5321},
        {"id": 2, "label_id": 2, "isthing": True,  "area": 4100},
        {"id": 0, "label_id": 55,"isthing": False, "area": 90000},
        ]

         id : seg_map에 들어있는 segment id
   label_id : 실제 클래스 id (예: person, car, road)
    isthing : thing인지(stuff가 아닌지)
       area : 픽셀 면적(얼마나 큰지)
    '''

    # 마스크 크기 준비. (이 픽셀은 ??? 이다, 를 판별하기 위한거임.)
    # color_mask: (H, W, 3)짜리 “컬러 이미지”를 0으로 만들어 둠(검은 화면)
    h, w = seg_map.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8) # 이미지는 보통 픽셀값이 0~255 범위고, 그 타입이 uint8

    rng = np.random.default_rng(42) #랜덤색을 동일하게 만들기 위해 시드 고정.

    # id -> color
    '''
    id_to_color = {
  0: [ 10, 200,  10],   # road
  1: [200,  30, 220],   # person #1
  2: [ 30,  60, 240],   # car #1}      #맨위 5*5 숫자맵에서 0,1,2값 참고

    '''
    id_to_color = {}  # 앞으로 “id → 색” 매핑을 담을 빈 박스 준비

    for s in segments_info:
        sid = int(s['id'])  #세그멘트 목록을 보며 정보 꺼내기.



        if sid not in id_to_color: # 위에서 뽑은 id값에 대한 색이 존재하지 않는다면,
            id_to_color[sid] = rng.intgers(0, 256, size = 3, dtype = np.unit8) # RGB :  rng.integers(0, 256, size=3)→ 0~255 사이 정수 3개를 뽑아 [R, G, B]로 쓰겠다.
            '''
            예를 들어 rng가 이런 값을 뽑았다 치자:
            id=0 → [12, 200, 33]
            id=1 → [210, 50, 140]
            id=2 → [30, 60, 240]

                            {
                0: [ 12, 200,  33],
                1: [210,  50, 140],
                2: [ 30,  60, 240],
                    }
            '''


    for sid, col in id_to_color.items():
        color_mask[seg_map == sid] = col #

        ''' 진짜 그림이 나오는 순간 : seg_map == sid
            seg_map == 1 하면 사람(#1) 위치는 True, 나머지는 False 인 불리언 마스크가 만들어짐.

            ex) sid=1
            False False False False False
            False True  True  True  False
            False True  True  True  False
            False False False False False
            False False False False False 그럼 True 인 부분만 색을 칠함.
            즉, 사람 영역이 한 번에 “보라색(예: [210,50,140])”으로 칠해지는 거임.
            최종적으로 road 영역은 초록 계열, person 영역은 보라 계열, car 영역은 파랑 계열 같이 색깔로 구분되는 그림이 됨.
        '''

    # # RGB -> BGR
    color_mask_bgr = cv2.cvtColor(color_mask, cv2.COLOR_RGB2BGR)


    # 영역을 반투명하게 해서 객체와 함께 표기, 원본영상이 완전혀 가려지지 않음 (픽셀단위 선형결합 : 같은 위치의 픽셀끼리 색 값을 비율로 섞는 것)
    overlay = cv2.addWeighted(frame_bgr, 1 - alpha, color_mask_bgr, alpha, 0) #파라미터 순서대로 : 원본영상, 원본 가중치, 색칠된 마스크, 마스크 가중치.
    return overlay, segments_info


In [11]:
# 프레임별로 중요한 세그먼트에 대하여 텍스트로 찍어주는 함수

def draw_topk_labels(frame_bgr, segments_info, topk=8):

    '''
    frame_bgr : OpenCV 영상 프레임(BGR)
    segments_info : panoptic 결과의 메타데이터 리스트 (각 segments 설명)
    topk : 최대 보여지는 텍스트(8개로 정했다) // 수정가능
    '''

    segs = sorted(segments_info,
                  key = lambda x : x.get('area', 0),
                  reverse = True)[:topk]


    '''
    segments_info를 면적 기준으로 큰 것부터 정렬 (앞에서 topk개 만큼 선택)
    '''




    y = 30 # 텍스트를 어디에 쓸지(현재 y축만 기준으로 잡음, 줄마다 증가시킬 예정)

    for s in segs:

        '''
        segs dict 생김새는 대충 이렇다.
                {
        "id": 1,
        "label_id": 0,
        "isthing": True,
        "area": 5321
        }
        '''

        # 클래스 이름 가져오기.
        # label_id는 숫자 클래스 id (예: 0)
        '''
        id2label = {
                0: "person",
                1: "bicycle",
                 2: "car",...}

                 label_id = 0 → "person" // 만약 매핑이 없으면 → "0" (문자열)
        '''
        label = id2label.get(int(s['label_id']), str(s['label_id']))

        # 화면에 찍을 문장 만들기.
        txt = f'{label} (thing = {s.get("isthing", None)})'
        '''
        결과 예시.
        person (thing = True)
        car (thing = True)
        road (thing = False) // 이 부분에서 무슨 클래스이고, thing인지, stuff인지 알랴줌. (thing : 개체로 셀 수 있는 것 (차, 말 등등)/ stuff로 퍼져있는 것(도로, 하늘 등등))
        '''

        # 실제로 화면에 글자 그리기.
        cv2.putText(frame_bgr, txt , (10, y), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 255, 255), 2, cv2.LINE_AA)
        '''

        putText : 파라미터 해석.
        (10, y) → 좌측 상단 위치
        FONT_HERSHEY_SIMPLEX → 기본 폰트
        0.7 → 글자 크기
        (255,255,255) → 흰색
        2 → 두께
        LINE_AA → 안티앨리어싱(부드럽게)
        '''

        y += 28

    return frame_bgr






#### mp4 영상 처리
> 입력 영상을 프레임 단위로 읽어서, </br>
  각 프레임에 panoptic segmentation을 적용하고, </br>
  결과를 새로운 파일로 저장한다. </br>

VideoCapture = “영상에서 프레임을 뽑는 기계”

cap.get(...) = “영상의 fps/크기/총프레임 같은 스펙 읽기”

VideoWriter_fourcc(*"mp4v") = “코덱 이름(mp4v)을 OpenCV가 쓰는 숫자 코드로 변환”

*는 "mp4v"를 'm','p','4','v'로 펼치는 언패킹

VideoWriter(...) = “프레임을 받아 mp4로 저장하는 기계”

cap.read() = “다음 프레임 주세요”

writer.write(frame) = “이 프레임을 출력 영상에 저장”

release() = “파일 닫고 저장 마무리”

----
개선가능 방안.
range(n) 대신 while read 루프 (frame_count 부정확한 파일도 안전)

처리 속도 위해 프레임 스킵(예: 매 2프레임마다 1번만 추론)

코덱이 안 먹히는 경우 대비해 ffmpeg 리먹스 자동화

In [ ]:
from tqdm import tqdm

in_video = "/content/video_test_in.mp4"      # 입력 영상
out_video = "/content/output_panoptic.mp4"  # 출력 영상


# cv2.VideoCapture : 나중에 프레임별로 한장씩 꺼낼 수 있게 만들어주는 객체.
cap = cv2.VideoCapture(in_video)

# isOpened() : 영상파일이 정상적으로 열렸는지 확인하기.
# 파일이 없으면: isOpened() == False → assert로 바로 멈춤.
assert cap.isOpened(), '비디오 경로 확인'


# cap.get(...) :초당 프레임수를 가져옴.
# 이 값이 ‘영상 데이터 자체’가 아니라 ‘영상을 설명하는 정보’라서 메타데이터라고 부름.
fps = cap.get(cv2.CAP_PROP_FPS)
'''
메타 데이터?
- 초당 몇 장인지 (FPS)
- 해상도가 얼마인지 (Width, Height)
- 총 프레임 수
- 영상 길이
- 코덱 정보 등
        ex)
        * fps = 30
        * width = 1920
        * height = 1080
        * frame_count = 4068

* 위 값으로는 영상 내용을 볼 수는 없으나, 영상파일을 어떻게 해석해야하는지 알 수 있다.
'''

# 영상 프레임의 가로 & 세로 크기를 가져옴.
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 영상의 총 프레임 수를 가져옴.
n = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 출력 영상 생성기 작성.
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
'''
'mp4v'와 같은 4글자 코덱 문자열을 opencv가 쓰는 정수 코드(fourcc 코드)로 변환해줌.
FOURCC = Four Character Code
(코덱을 4글자로 표현하는 관례)
* 을 넣은건, 연산자(unpack)로 이용하기 위함, -> cv2.VideoWriter_fourcc('m','p','4','v') 이렇게 리스트 또는 문자열을 펼쳐버림.
4글자 코덱 문자열”은 비디오 압축/인코딩 방식을 4개의 문자로 약속해 둔 ‘이름표’
이걸 컴퓨터가 빠르게 쓰기 위해 정수 코드(FourCC) 로 바꾼다.
'''

# 프레임을 지정한 코덱으로 저장하는 객체 생성.
writer = cv2.VideoWriter(out_video, fourcc, fps, (w, h))
'''
프레임을 받아서(writer.write(frame))mp4 파일로 차곡차곡 저장하는 객체.

* 인자 의미.
out_video: 저장 파일 경로
fourcc: 어떤 코덱으로 저장할지(위에서 만든 코드)
fps: 초당 몇 프레임으로 저장할지
(w, h): 프레임 크기(가로, 세로)
'''

# 프레임 처리 루프 (프레임 개수만큼)
for _ in tqdm(range(n)):
    cap.read()
    '''
    cap.read() 반환값 :
        - ret: 성공 여부 (True/False)
        - frame: 이미지 배열 (NumPy, BGR, shape=(h,w,3))
ex)
정상 프레임이면:
    -  ret=True
    - frame.shape == (1080, 1920, 3)

영상 끝이면:
    - ret=False
    - frame=None 비슷한 상태
    '''

    # 끝에 도달했거나 오류라면 반복을 종료
    if not ret:
        break


    overlay, segments_info = panoptic_overlay_bgr(frame, alpha = 0.55)
    '''
    frame 한 장에 대하여 panoptic segmentation 수행.
    결과를 색으로 칠해서 overlay 프레임을 만들고 반환함.

    반환값 :
        overlay: 원본 + 색칠된 결과가 합쳐진 프레임 (BGR, (h,w,3))
        segments_info: 세그먼트 메타정보 리스트

    ex)
    frame 안에 사람이 2명, 차 1대가 있다면
    overlay에는 사람/차/도로 등이 색으로 덧칠됨
    segments_info에는 person#1, person#2, car#1, road 같은 정보가 담김

    '''
    # 텍스트 라벨 붙이기.
    # egments_info에서 면적 큰 세그먼트 6개를 골라 overlay 프레임 좌측 상단에 텍스트로 표시
    overlay = draw_topk_labels(overlay, segments_info, topk = 6)


    # 프레임 저장.
    writer.write(overlay)
    '''
        하는 일
            - overlay 프레임을 출력 영상에 한 장 추가한다.

        예시로 머릿속 그림
        - 첫 번째 프레임 write
        - 두 번째 프레임 write
                 …
        - 4068번째 프레임 write
        → 이게 모여서 mp4가 됨
    '''

# 마무리 : 리소스 해제.
cap.release()
writer.release()
'''
파일 핸들을 닫아야 저장이 “완료”되고 파일이 정상 재생됨.
writer를 release 안 하면 mp4가 손상되거나, 파일이 비정상적으로 끝날 수 있어.
'''

# 출력 확인.
print("saved:", out_video) # 결과 경로 확인용


